# 09 - Post hoc amendment 1 - local recalibration of the transported models (RQ3)

**Post hoc.** Decided after notebook 05 showed that the frozen models rank participants well in the other cohort but
give miscalibrated probabilities. The question: does a local recalibration, estimated in the target cohort without
retraining the model, correct the probabilities and the decisions at the threshold of 0.5?

**How it is kept apart from the pre-specified analysis**
- config/analysis_plan.json is not edited: its hash stays the same and notebooks 00–08 are unaffected.
- The amendment has its own plan, config/post_hoc_plan.json, which states the hash of the plan it amends. This
  notebook freezes it in outputs/post_hoc/post_hoc_plan_frozen.json before computing anything and refuses to run
  if it changes later.
- No model is trained. The notebook reads only the predictions saved by notebook 05
  (outputs/predictions/transport_RQ3a.csv and transport_RQ3b.csv) and writes only to outputs/post_hoc/.
  outputs/results/ is not touched, so the readings of RQ1-RQ3 and the check of notebook 08 do not change.

**Method.** Logistic recalibration of the frozen model's probability p in the target cohort: intercept only,
logit(p') = a + logit(p), and intercept and slope, logit(p') = a + b·logit(p). Each participant's recalibrated
probability uses (a, b) estimated without that participant (leave-one-out). Calibration slope,
calibration-in-the-large and Brier as in notebook 05; bootstrap 95% intervals.

**Outputs:** outputs/post_hoc/ (Table S7 of the article, results, subject-level recalibrated predictions, summary).

**Time:** about a minute.

In [ ]:
# Setup: make the eegad package importable (also when it was not installed with `pip install -e .`)
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore", category=RuntimeWarning)   # MNE notices about head size and 'boundary' events
warnings.filterwarnings("ignore", category=FutureWarning)

_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config" / "analysis_plan.json").exists()), None)
if _root is None:
    raise FileNotFoundError("Open this notebook from inside the repository (eeg-ad-demographics/notebooks).")
if str(_root / "src") not in sys.path:
    sys.path.insert(0, str(_root / "src"))

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display

from eegad.config import load_context
from eegad.data import load_table
from eegad.posthoc import (METHODS, RECALIBRATION_NOTE, freeze_post_hoc_plan, recalibration_analysis,
                           recalibration_table)
from eegad.reporting import read_json, write_json

ctx = load_context("09")              # also checks that the analysis plan is the one frozen by notebook 00
out = ctx.out
PH, PH_HASH, PH_FROZEN = freeze_post_hoc_plan(ctx)
N_BOOT = ctx.rt["n_boot"] if ctx.quick else int(PH["metrics"]["n_boot"])
SEED, THR, N_BINS = int(PH["metrics"]["seed"]), float(PH["metrics"]["threshold"]), int(PH["metrics"]["calibration_curve"]["bins"])
FOLDER = out / "post_hoc"
TRANSPORT = read_json(out / "results" / "transport.json")["TRANSP"]
print(f"Amendment {PH['amendment']}: {PH['title']}\nbootstrap {N_BOOT} | threshold {THR} | seed {SEED}")

## Recalibration in each direction

In [ ]:
RESULTS, SUBJ = {}, {}
for key, label in (("RQ3a", "BrainLat → Greek"), ("RQ3b", "Greek → BrainLat")):
    pred = load_table(out / "predictions" / f"transport_{key}.csv")
    res, subj = recalibration_analysis(pred, label, THR, N_BOOT, SEED, N_BINS)
    # the frozen-model numbers must be those of notebook 05 (same predictions, same definitions)
    t5 = TRANSPORT[key]
    fz = res["versions"]["frozen"]
    assert np.isclose(fz["calibration_slope"], t5["calibration_slope"]) and np.isclose(fz["citl"], t5["citl"]) \
        and np.isclose(fz["brier"], t5["brier"]), f"{key}: the predictions differ from those of notebook 05"
    RESULTS[key], SUBJ[key] = res, subj
    subj.to_csv(FOLDER / f"recalibrated_predictions_{key}.csv", index=False)
    print(f"\n{label} (n = {fz['n']}: {fz['n_AD']} AD, {fz['n_HC']} HC)")
    for v in ("frozen",) + METHODS:
        m = res["versions"][v]
        eq = res["equations"].get(v)
        print(f"  {v:20s} slope {m['calibration_slope']:5.2f} | CITL {m['citl']:5.2f} | Brier {m['brier']:.3f} "
              f"| sens {m['metrics']['sens']:.2f} spec {m['metrics']['spec']:.2f}"
              + (f" | whole-cohort recalibration: a = {eq['a']:.2f}, b = {eq['b']:.2f}" if eq else ""))

## Table S7 and summary

In [ ]:
S7 = recalibration_table(RESULTS)
display(S7)
S7.to_csv(FOLDER / "recalibration_table.csv", index=False, encoding="utf-8-sig")
(FOLDER / "recalibration_note.txt").write_text(RECALIBRATION_NOTE.format(n_boot=N_BOOT), encoding="utf-8")
write_json(FOLDER / "recalibration.json", {"amendment": PH["amendment"], "post_hoc_plan_hash": PH_HASH,
                                           "frozen_at": PH_FROZEN, "amends_plan_hash": ctx.plan_hash,
                                           "mode": ctx.tag, "n_boot": N_BOOT, "results": RESULTS})
lines = [f"# Post hoc amendment {PH['amendment']}: local recalibration (NOT pre-specified)", "",
         f"Amendment hash {PH_HASH}, frozen on {PH_FROZEN}; amends the analysis plan {ctx.plan_hash}. "
         f"Mode: {ctx.tag}.", ""]
for key, res in RESULTS.items():
    fz, ab = res["versions"]["frozen"], res["versions"]["intercept_and_slope"]
    d = ab["brier"] - fz["brier"]
    lines.append(f"- {res['label']}: calibration slope {fz['calibration_slope']:.2f} -> {ab['calibration_slope']:.2f}; "
                 f"calibration-in-the-large {fz['citl']:.2f} -> {ab['citl']:.2f}; Brier {fz['brier']:.3f} -> "
                 f"{ab['brier']:.3f} (change {d:+.3f} [{ab['ci']['delta_brier'][0]:+.3f}, {ab['ci']['delta_brier'][1]:+.3f}]); "
                 f"sensitivity {fz['metrics']['sens']:.2f} -> {ab['metrics']['sens']:.2f}, specificity "
                 f"{fz['metrics']['spec']:.2f} -> {ab['metrics']['spec']:.2f} (intercept and slope, leave-one-out)")
(FOLDER / "post_hoc_summary.md").write_text("\n".join(lines) + "\n", encoding="utf-8")
print("\n".join(lines))